In [1]:
import os
from pyspark.sql import SparkSession

# We must declare the packages Spark needs to download to talk to Delta and MinIO (S3)
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages io.delta:delta-spark_2.12:3.1.0,org.apache.hadoop:hadoop-aws:3.3.4 pyspark-shell'

spark = SparkSession.builder \
    .appName("LakehouseSetup") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print("Connected successfully. Ready to build the Lakehouse.")

Spark Version: 3.5.0
Connected successfully. Ready to build the Lakehouse.


In [12]:
from pyspark.sql.functions import col, lower, translate, to_timestamp, lower
from pyspark.sql.utils import AnalysisException

### olist_customers_dataset.csv Bronze to Silver

In [8]:
# 1. LOAD BRONZE DATA
df_customers_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("s3a://olist-data/bronze/olist_customers_dataset.csv")

df_customers_bronze.printSchema()
df_customers_bronze.show(5)

# 2. TRANSFORM & CLEAN (Standardization)
# Lowercase the city and remove common Portuguese accents
df_cleaned = df_customers_bronze \
    .withColumn("customer_city", lower(col("customer_city"))) \
    .withColumn("customer_city", translate(col("customer_city"), "áàâãäéèêëíìîïóòôõöúùûüç", "aaaaaeeeeiiiiooooouuuuc"))

# 3. AUTOMATED DATA QUALITY (DQ) CHECKS
print("Running Automated DQ Checks...")

# Rule A: Customer ID cannot be null
print("DQ Check: If there is any NULL value in customer_id ...")
null_id_count = df_cleaned.filter(col("customer_id").isNull()).count()
if null_id_count > 0:
    raise Exception(f"DQ FAIL: Found {null_id_count} rows with a null customer_id. Pipeline halted.")

# Rule B: Customer ID must be unique
print("DQ Check: If there are any duplicated value in customer_id ...")
total_rows = df_cleaned.count()
unique_ids = df_cleaned.select("customer_id").distinct().count()
if total_rows != unique_ids:
    raise Exception(f"DQ FAIL: Found {total_rows - unique_ids} duplicate customer_ids. Pipeline halted.")

print("DQ Checks Passed! Proceeding to write to Silver.")

# 4. WRITE TO SILVER (Enforces Schema & Enables Time Travel)
df_cleaned.write.format("delta") \
    .mode("overwrite") \
    .save("s3a://olist-data/silver/customers")

print("Successfully wrote Customers to Silver Delta Table.")

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83

### olist_orders_dataset.csv Bronze to Silver

In [9]:
# 1. LOAD BRONZE DATA
df_orders_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("s3a://olist-data/bronze/olist_orders_dataset.csv")

# 2. TRANSFORM & CLEAN
# Whitelist approach: Only keep successfully delivered orders
# delivered, shipped, canceled, unavailable, invoiced, processing, created, and approved
df_cleaned = df_orders_bronze.filter(col("order_status") == "delivered")

# Convert string dates to actual PySpark Timestamps
df_cleaned = df_cleaned.withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp"))) \
                       .withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date")))

# 3. DQ CHECKS
print("Running DQ Checks...")
# Check for null primary keys
print("DQ Check: If there is any NULL value in order_id ...")
null_orders = df_cleaned.filter(col("order_id").isNull()).count()
if null_orders > 0:
    raise Exception(f"DQ FAIL: Found {null_orders} null order_ids! Pipeline halted.")

print("DQ Checks Passed! Proceeding to write to Silver.")

# 4. WRITE TO SILVER (Delta format)
df_cleaned.write.format("delta") \
    .mode("overwrite") \
    .save("s3a://olist-data/silver/orders")

print("Successfully wrote Orders to Silver Delta Table.")

Running DQ Checks...
DQ Check: If there is any NULL value in order_id ...
DQ Checks Passed! Proceeding to write to Silver.
Successfully wrote Orders to Silver Delta Table.


### olist_order_items_dataset.csv Bronze to Silver

In [10]:
# 1. LOAD BRONZE
df_items_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("s3a://olist-data/bronze/olist_order_items_dataset.csv")

# 2. TRANSFORM & CLEAN
# Ensure price and freight are numbers, and filter out impossible negative values
df_items_clean = df_items_bronze \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double")) \
    .filter(col("price") >= 0)

# 3. DQ CHECKS
print("DQ Check: If there is any NULL value in neither order_id nor product_id ...")
null_foreign_keys = df_items_clean.filter(col("order_id").isNull() | col("product_id").isNull()).count()
if null_foreign_keys > 0:
    raise Exception(f"DQ FAIL: Found {null_foreign_keys} items missing order/product IDs!")
print("DQ Checks Passed! Proceeding to write to Silver.")

# 4. WRITE TO SILVER
df_items_clean.write.format("delta") \
    .mode("overwrite") \
    .save("s3a://olist-data/silver/order_items")

print("Successfully wrote Order Items to Silver.")

DQ Check: If there is any NULL value in neither order_id nor product_id ...
DQ Checks Passed! Proceeding to write to Silver.
Successfully wrote Order Items to Silver.


### olist_products_dataset.csv Bronze to Silver

In [13]:
# 1. LOAD BRONZE
df_products_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("s3a://olist-data/bronze/olist_products_dataset.csv")

# 2. TRANSFORM & CLEAN
# Lowercase the category names and replace NULLs with 'unknown' to prevent data loss
df_products_clean = df_products_bronze \
    .withColumn("product_category_name", lower(col("product_category_name"))) \
    .fillna("unknown", subset=["product_category_name"])

# 3. DQ CHECKS
print("DQ Check: If there is any NULL value in product_id ...")
null_product_ids = df_products_clean.filter(col("product_id").isNull()).count()
if null_product_ids > 0:
    raise Exception(f"DQ FAIL: Found {null_product_ids} null product_ids!")
print("DQ Checks Passed! Proceeding to write to Silver.")

# 4. WRITE TO SILVER
df_products_clean.write.format("delta") \
    .mode("overwrite") \
    .save("s3a://olist-data/silver/products")

print("Successfully wrote Products to Silver.")


--- Starting Products Silver Pipeline ---
DQ Check: If there is any NULL value in product_id ...
DQ Checks Passed! Proceeding to write to Silver.
Successfully wrote Products to Silver.


In [14]:
from pyspark.sql.functions import col, sum, countDistinct, max, round

print("--- Starting Gold Layer: Customer 360 ---")

# 1. LOAD SILVER DATA (No schemas needed, Delta remembers!)
df_customers = spark.read.format("delta").load("s3a://olist-data/silver/customers")
df_orders = spark.read.format("delta").load("s3a://olist-data/silver/orders")
df_items = spark.read.format("delta").load("s3a://olist-data/silver/order_items")

# 2. THE BIG JOIN (Fusing the data together)
# Join Orders and Items to figure out the cost of each order
df_order_spend = df_orders.join(df_items, on="order_id", how="inner")

# Join with Customers to attach the real human ID and their city
df_full_history = df_order_spend.join(df_customers, on="customer_id", how="inner")

# 3. AGGREGATE THE CDP (Squash it down to one row per human)
# Notice we group by customer_UNIQUE_id here!
df_customer_360 = df_full_history.groupBy("customer_unique_id", "customer_city").agg(
    
    # Frequency: Count how many unique orders they placed
    countDistinct("order_id").alias("total_orders"),
    
    # Monetary: Sum of all item prices + freight costs, rounded to 2 decimals
    round(sum(col("price") + col("freight_value")), 2).alias("total_lifetime_value"),
    
    # Recency: Find the timestamp of their most recent purchase
    max("order_purchase_timestamp").alias("last_purchase_date")
)

# 4. WRITE TO GOLD
df_customer_360.write.format("delta") \
    .mode("overwrite") \
    .save("s3a://olist-data/gold/customer_360")

print("Success! Customer 360 Table built in Gold Layer.")

# Let's peek at the final product! Sort by highest spenders.
df_customer_360.orderBy(col("total_lifetime_value").desc()).show(5)

--- Starting Gold Layer: Customer 360 ---
Success! Customer 360 Table built in Gold Layer.
+--------------------+--------------+------------+--------------------+-------------------+
|  customer_unique_id| customer_city|total_orders|total_lifetime_value| last_purchase_date|
+--------------------+--------------+------------+--------------------+-------------------+
|0a0a92112bd4c708c...|rio de janeiro|           1|            13664.08|2017-09-29 15:24:52|
|da122df9eeddfedc1...|      araruama|           2|             7571.63|2017-04-01 15:58:41|
|763c8b1c9c68a0229...|    vila velha|           1|             7274.88|2018-07-15 14:49:44|
|dc4802a71eae9be1d...|  campo grande|           1|             6929.31|2017-02-12 20:37:36|
|459bef486812aa252...|       vitoria|           1|             6922.21|2018-07-25 18:10:17|
+--------------------+--------------+------------+--------------------+-------------------+
only showing top 5 rows

